# Pretrain — Scratch Shakespeare Transformer

PyTorch + Lightning. See `README.md` for architecture, BPC metric, and workflow docs.

## Imports

In [9]:
import torch
import wandb
import os
#import torch.nn as nn
import lightning as L
#from utils import log_results, save_model
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import WandbLogger

In [2]:
from model import Config, CausalSelfAttention, TransformerBlock, Transformer
from data import train_tokenizer, load_tokenizer, make_dataloaders

In [3]:
cfg = Config(
    vocab_size  = 1000,
    B           = 32,
    n           = 256,
    d_model     = 384,
    d_ff        = 1536,
    H           = 8,
    layers      = 6,
    lr          = 3e-4,
    drop_prob   = 0.1,
    epochs      = 100,
)

## Training



In [4]:
class ShakespeareModule(L.LightningModule):
    """LightningModule wrapping Transformer for use with Trainer.

    training_step and validation_step log loss to wandb via self.log().
    configure_optimizers returns AdamW + ReduceLROnPlateau, matching transformer.ipynb.
    """
    def __init__(self, cfg):
        super().__init__()
        self.model = Transformer(cfg)
        self.cfg = cfg
        self.loss_fn = torch.nn.CrossEntropyLoss()
        

    def training_step(self, batch, _):
        """Forward pass on one batch; return loss. self.log() sends to wandb."""
        logits, targets = self.model(batch) #nn.Module says when you call a module, it calls the "forward" function
        loss = self.loss_fn(logits.reshape(-1, self.cfg.vocab_size), targets.reshape(-1))
        self.log("train_loss", loss, on_epoch=True, on_step=False) #defined in L.LightningModule
        return loss
        

    def validation_step(self, batch, _):
        """Same as training_step but no gradient (handled by Lightning). Logged as val_loss."""
        logits, targets = self.model(batch)
        loss = self.loss_fn(logits.reshape(-1, self.cfg.vocab_size), targets.reshape(-1))
        self.log("val_loss", loss, on_epoch=True)
        

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.cfg.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }

### Callbacks to print progress to the notebook, and run() command gets everything ready for the lightning trainer 

In [5]:
class EpochPrinter(L.Callback):  ##AI Slop
    def __init__(self):
        self._train_losses = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs
        self._train_losses.append(loss.item())
        n_batches = trainer.num_training_batches
        # \r overwrites the same line each batch so output doesn't scroll
        pl_module.print(f"Epoch {trainer.current_epoch} | "
                        f"Loss {loss.item():.4f} | "
                        f"Batch {batch_idx + 1} / {n_batches}",
                        end="\r", flush=True)

    def on_validation_epoch_end(self, trainer, pl_module):
        train_loss = sum(self._train_losses) / len(self._train_losses) if self._train_losses else float("nan")
        self._train_losses = []
        val_loss = trainer.callback_metrics.get("val_loss", float("nan"))
        line = f"Epoch {trainer.current_epoch:>3}  train: {train_loss:.4f}  val: {val_loss:.4f}"
        pl_module.print(f"\r{line.ljust(60)}", flush=True)


class StopOnFile(L.Callback):
    # Touch this file from a terminal to stop training cleanly at end of current epoch
    STOP_FILE = "/teamspace/studios/this_studio/shakespeare_generator/stop_training"

    def on_train_epoch_end(self, trainer, pl_module):
        if os.path.exists(self.STOP_FILE):
            trainer.should_stop = True
            os.remove(self.STOP_FILE)


def run(cfg, train_path, val_path, note=""):
    """Tokenize → dataloaders → train. Returns (trainer, module, tokenizer)."""

    tokenizer_path = "tokenizer.json"

    # Load existing tokenizer if vocab_size matches, otherwise retrain
    try:
        tokenizer = load_tokenizer(path=tokenizer_path)
        assert len(tokenizer.get_vocab()) == cfg.vocab_size
    except:
        tokenizer = train_tokenizer(train_path=train_path, vocab_size=cfg.vocab_size, save_path=tokenizer_path)

    train_loader = make_dataloaders(path=train_path, tokenizer=tokenizer, cfg=cfg, shuffle=True)
    val_loader = make_dataloaders(path=val_path, tokenizer=tokenizer, cfg=cfg, shuffle=False)

    module = ShakespeareModule(cfg)

    trainer = L.Trainer(
        max_epochs=cfg.epochs,
        callbacks=[
            EpochPrinter(),
            StopOnFile(),
            ModelCheckpoint(monitor="val_loss", save_top_k=1, filename="best"),  # saves best checkpoint automatically
            EarlyStopping(monitor="val_loss", patience=4),
        ],
        logger=WandbLogger(project="shakespeare"),
        accelerator="gpu",
        devices=1,
        gradient_clip_val=1.0,
        enable_progress_bar=False,  # disabled — EpochPrinter replaces it; tqdm causes severe output lag in VS Code
    )
    trainer.fit(module, train_loader, val_loader)

    return trainer, module, tokenizer

In [6]:
trainer, module, tokenizer = run(
    cfg,
    train_path="data/full_train_minus_tiny_val.txt",
    val_path="data/full_val.txt",
)
wandb.finish()
# run this command in the terminal to end the train

# touch /teamspace/studios/this_studio/shakespeare_generator/stop_training

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /teamspace/studios/this_studio/.netrc.
wandb: Currently logged in as: mattcecil (mattcecil_personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ Transformer      │ 11.0 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 11.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.0 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 84                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Epoch   0  train: nan  val: 338.6412                        
Epoch   0  train: 18.2781  val: 5.9804                      
Epoch   1  train: 5.3173  val: 4.8245                       
Epoch   2  train: 4.6861  val: 4.5282                       
Epoch   3  train: 4.4058  val: 4.3204                       
Epoch   4  train: 4.1924  val: 4.2190                       
Epoch   5  train: 4.0269  val: 4.1029                       
Epoch   6  train: 3.8982  val: 4.0168                       
Epoch   7  train: 3.7928  val: 3.9574                       
Epoch   8  train: 3.6949  val: 3.9012                       
Epoch   9  train: 3.6145  val: 3.8660                       
Epoch  10  train: 3.5440  val: 3.8825                       
CSV row appended to training_log.csv
Saved model to checkpoints/cookies-n-cream-tart-10.pt


('checkpoints/cookies-n-cream-tart-10.pt', 'cookies-n-cream-tart-10')

In [10]:
wandb.finish()


from eval import evaluate
from utils import log_results, save_model

tiny_val_path = "data/tiny_val.txt"
best_path = trainer.checkpoint_callback.best_model_path
wandb_name = trainer.logger.experiment.name

module = ShakespeareModule.load_from_checkpoint(best_path, cfg=cfg)
module.model.to("cuda")

tiny_val_loss = evaluate(module.model, tiny_val_path, tokenizer, source_type="custom", cfg=cfg)

with open(tiny_val_path) as f:
    tiny_val_text = f.read()
avg_chars_per_token = len(tiny_val_text) / len(tokenizer.encode(tiny_val_text).ids)
val_loss = trainer.checkpoint_callback.best_model_score.item()

log_results(cfg, train_loss=0.0, val_loss=val_loss, tiny_val_loss=tiny_val_loss,
            avg_chars_per_token=avg_chars_per_token, name=wandb_name,
            csv_path="training_log.csv")
save_model(module.model, cfg, directory="checkpoints", name=wandb_name)

epoch,▁▁▂▂▂▂▃▃▄▄▅▅▅▅▆▆▇▇▇▇██
train_loss,█▂▂▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▂▂▂▂▃▃▄▄▅▅▅▅▆▆▇▇▇▇██
val_loss,█▄▃▃▂▂▁▁▁▁▁
epoch,10
train_loss,3.54398
trainer/global_step,1825
val_loss,3.88246


CSV row appended to training_log.csv
Saved model to checkpoints/cookies-n-cream-tart-10.pt


('checkpoints/cookies-n-cream-tart-10.pt', 'cookies-n-cream-tart-10')